===== Cell 1: 基础环境配置与 GPU 绑定 =====

In [ ]:
# =========================================================
# Cell 1: 环境配置与模型导入
# =========================================================
import os
import sys
import warnings

# 1. 强制只使用 GPU 0 (必须在 import torch 之前)
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
warnings.filterwarnings("ignore")

import torch
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import scipy.sparse as sp
import decoupler as dc
from io import StringIO
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# 2. 导入原版 NicheCompass
LOCAL_SRC = "/home/zhangjunyi/xiangmu/nichecompass-main/src"
if LOCAL_SRC not in sys.path:
    sys.path.insert(0, LOCAL_SRC)
import nichecompass as nc

print("✅ CUDA 可用状态:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ 当前使用 GPU:", torch.cuda.get_device_name(0))
print("✅ NicheCompass 路径:", nc.__file__)

===== Cell 2: Step 1 - 构建代谢知识库与加载数据 =====

In [ ]:
# =========================================================
# Cell 2: 知识库提取与数据加载 (Step 1)
# =========================================================
print("=== Step 1: 构建代谢通讯轴四元组知识库 ===")

# 1. 解析你的四元组知识库
tmcn_csv = """TMCN_Name,Source_Pathways,Source_Genes,Target_Genes,Biologic_Meaning
TMCN_Lactate_Axis,"EGFR,PI3K,Hypoxia","HK2,LDHA,LDHB,SLC16A3","SLC16A1,HCAR1",乳酸_肿瘤酸化与代谢重生
TMCN_Adenosine_Axis,"Hypoxia","MYC,ENTPD1,NT5E","ADORA1,ADORA2A,ADORA2B,ADORA3",腺苷_强效免疫抑制与血管生成
TMCN_PGE2_Axis,"NFkB,JAK-STAT","PTGS2,PTGES","PTGER1,PTGER2,PTGER4",前列腺素E2_炎症微环境重塑
TMCN_Glutamine_Axis,"JAK-STAT,TGFb","MYC,GLUL","SLC1A5,SLC38A2",谷氨酰胺_基质-肿瘤代谢共生(寄生)
TMCN_Succinate_Axis,"Hypoxia","EGLN1,TET2,SLC25A10",SUCNR1,琥珀酸_琥珀酸介导的巨噬细胞极化
"""
axis_table = pd.read_csv(StringIO(tmcn_csv))
AXES =[x.replace("TMCN_", "").replace("_Axis", "") for x in axis_table["TMCN_Name"]]

# 辅助函数：解析基因列表
def split_items(x):
    return[i.strip() for i in str(x).split(",") if i.strip()]

def get_valid_genes(adata, genes):
    return[g for g in genes if g in adata.var_names]

# 2. 加载原始空间转录组数据
file_path = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_integrated.h5ad"
adata = sc.read_h5ad(file_path)

print(f"✅ 数据加载成功: {adata.n_obs} spots, {adata.n_vars} genes")
display(axis_table)

===== Cell 3: Step 2 - 计算并解耦连续分数 (不输入给模型) =====


In [ ]:
# =========================================================
# Cell 3: 连续功能分数推断与解耦 (Step 2)
# =========================================================
print("=== Step 2: 计算 Spot 级连续分数 (绝对无监督解耦) ===")

# 1. 运行 PROGENy 提取 Pathway_score
print("--> 正在计算 Pathway_score (PROGENy)...")
net_progeny = dc.get_progeny(organism="human", top=500)
dc.run_mlm(mat=adata, net=net_progeny, source="source", target="target", weight="weight", verbose=False, use_raw=False)
df_pathway = dc.get_acts(adata, obsm_key="mlm_estimate").to_df()
df_pathway.columns =[c.strip().replace("-", "_").replace(" ", "_") for c in df_pathway.columns]

# 2. 运行 AUCell 提取 Enzyme_score
print("--> 正在计算 Enzyme_score (AUCell)...")
enzyme_records =[]
for _, row in axis_table.iterrows():
    ax = row["TMCN_Name"].replace("TMCN_", "").replace("_Axis", "")
    for g in split_items(row["Source_Genes"]):
        if g in adata.var_names:
            enzyme_records.append({"source": ax, "target": g})
dc.run_aucell(mat=adata, net=pd.DataFrame(enzyme_records), source="source", target="target", min_n=1, verbose=False, use_raw=False)
df_enzyme = dc.get_acts(adata, obsm_key="aucell_estimate").to_df()

# 3. 对通路和酶进行 MinMax 归一化 (0~1)
scaler = MinMaxScaler()
df_pathway_scaled = pd.DataFrame(scaler.fit_transform(df_pathway), index=df_pathway.index, columns=df_pathway.columns)
df_enzyme_scaled = pd.DataFrame(scaler.fit_transform(df_enzyme), index=df_enzyme.index, columns=df_enzyme.columns)

# 4. 提取 Receptor_score 并计算 Sender & Receiver 最终分数
print("--> 正在计算 Sender_score 与 Receiver_score...")
def calc_receptor_score(genes):
    valid = get_valid_genes(adata, genes)
    if not valid: return np.zeros(adata.n_obs)
    X_sub = adata[:, valid].X
    mean_exp = np.asarray(X_sub.mean(axis=1)).reshape(-1) if sp.issparse(X_sub) else np.asarray(X_sub).mean(axis=1).reshape(-1)
    return scaler.fit_transform(mean_exp.reshape(-1, 1)).flatten()

for _, row in axis_table.iterrows():
    ax = row["TMCN_Name"].replace("TMCN_", "").replace("_Axis", "")
    pathways =[p.strip().replace("-", "_").replace(" ", "_") for p in split_items(row["Source_Pathways"])]
    target_genes = split_items(row["Target_Genes"])
    
    # 提取存在的通路
    valid_paths =[p for p in pathways if p in df_pathway_scaled.columns]
    
    # Sender = Pathway (均值) * Enzyme
    p_score = df_pathway_scaled[valid_paths].mean(axis=1).values
    e_score = df_enzyme_scaled[ax].values
    adata.obs[f"{ax}_Sender_Score"] = p_score * e_score
    
    # Receiver = Receptor 表达量归一化
    adata.obs[f"{ax}_Receiver_Score"] = calc_receptor_score(target_genes)

print("✅ 所有连续分数提取完毕！保存在 adata.obs 中。")
# 预览前5行打分
score_cols =[c for c in adata.obs.columns if "Sender_Score" in c or "Receiver_Score" in c]
display(adata.obs[score_cols].head())

===== Cell 4: Step 3 - 无监督训练与生成“微地块” =====
注意：这段代码中，NicheCompass 只看 高变基因和基础空间图，完全屏蔽了刚才算出来的代谢特征！

In [ ]:
# =========================================================
# Cell 4: NicheCompass 无监督拓扑学习与过聚类 (Step 3)
# =========================================================
print("=== Step 3: NicheCompass 构建物理联通图与微地块骨架 ===")

# 1. 提取 HVG (Top 2000) 加上四元组特征基因，确保组织底色干净
print("--> 提取高变基因 (HVG)...")
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3" if "counts" in adata.layers else "cell_ranger")
hvg_genes = set(adata.var_names[adata.var['highly_variable']])
forced_genes = set()
for _, row in axis_table.iterrows():
    forced_genes.update(split_items(row["Source_Genes"]) + split_items(row["Target_Genes"]))
keep_genes = sorted(hvg_genes.union(forced_genes.intersection(adata.var_names)))
adata_model = adata[:, keep_genes].copy()

# 2. 构建基础物理联通图 (NicheCompass GCN 使用)
print("--> 构建基于距离的空间物理联通图 (KNN)...")
sq.gr.spatial_neighbors(adata_model, coord_type="generic", spatial_key="spatial", n_neighs=8)

# 3. 加载通用 LR 先验 (本地 NicheNet)
print("--> 加载通用 LR 先验 (NicheNet)...")
cache_dir = "/home/zhangjunyi/xiangmu/nichecompass-main/cache_nichenet"
gp_dict = nc.utils.extract_gp_dict_from_nichenet_lrt_interactions(
    species="human", version="v2",
    keep_target_genes_ratio=0.25, max_n_target_genes_per_gp=50,
    load_from_disk=True,
    lr_network_file_path=f"{cache_dir}/nichenet_lr_network.csv",
    ligand_target_matrix_file_path=f"{cache_dir}/nichenet_ligand_target_matrix.csv"
)
nc.utils.add_gps_from_gp_dict_to_adata(gp_dict=gp_dict, adata=adata_model)

# 4. 纯无监督初始化模型 (无 custom edge weights, 无 node features)
print("--> 初始化并训练 NicheCompass 模型 (完全无监督)...")
model = nc.models.NicheCompass(
    adata=adata_model,
    counts_key="counts" if "counts" in adata_model.layers else None,
    adj_key="spatial_connectivities", # 纯物理邻接图
    conv_layer_encoder="gcnconv"
)

# 开始训练
model.train(n_epochs=50, n_epochs_all_gps=10, edge_batch_size=128, node_batch_size=256, use_cuda_if_available=True)

# 5. 提取 Latent 并进行“过聚类” (Over-clustering)
print("--> 提取 Latent 表征并生成微地块 (Micro-clusters)...")
adata_model.obsm["nichecompass_latent"] = model.get_latent_representation()

# 构建隐空间邻接图
sc.pp.neighbors(adata_model, use_rep="nichecompass_latent", key_added="latent_neighbors")

# 关键：高分辨率聚类 (resolution=1.5)，把组织切碎
sc.tl.leiden(adata_model, neighbors_key="latent_neighbors", key_added="micro_clusters", resolution=1.5)

print(f"✅ 无监督微地块骨架构建完成！共切分出 {adata_model.obs['micro_clusters'].nunique()} 个微地块。")

# （可选保存）
# adata_model.write_h5ad("/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_Stage1_Unsupervised.h5ad")